In [ ]:
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE_DIR = "/content/drive/MyDrive/running_coach"
sys.path.insert(0, f"{BASE_DIR}/tools")
from calculate_pace_zones import classify_workouts, format_pace
from parse_workout_data import load_workouts

workouts = load_workouts(f"{BASE_DIR}/data/processed/workouts_normalized.json", days=90)
enriched = classify_workouts(workouts)

structured  = [w for w in enriched if w["classification"]["is_structured"]]
needs_input = [w for w in enriched if w["classification"]["needs_input"]]
print(f"Structured: {len(structured)} | Needs input: {len(needs_input)}")

Mounted at /content/drive
Structured: 24 | Needs input: 0


In [ ]:
from parse_workout_data import load_workouts

workouts = load_workouts(f"{BASE_DIR}/data/processed/workouts_normalized.json", days=90)

# Print all activity names so we can see the actual format
for w in sorted(workouts, key=lambda x: x['date'], reverse=True)[:20]:
    print(f"{w['date']} | {w.get('garmin_name', w.get('name', ''))}")

2026-04-26 | Halifax Running
2026-04-25 | Halifax - 2026-04-25
2026-04-24 | Halifax Running
2026-04-23 | Halifax Running
2026-04-22 | Halifax - 2026-04-22
2026-04-21 | Halifax - Easy w/ strides
2026-04-19 | Halifax Running
2026-04-18 | Halifax - 2026-04-17
2026-04-18 | Halifax - 2026-04-17
2026-04-16 | Halifax Running
2026-04-16 | Halifax Running
2026-04-15 | Halifax - 2026-04-15
2026-04-14 | Halifax Running
2026-04-12 | Halifax Running
2026-04-11 | Morning Run
2026-04-10 | Halifax - 2026-04-10
2026-04-08 | Halifax Running
2026-04-07 | Halifax Running
2026-04-05 | Halifax - Easy w/ strides
2026-04-05 | Halifax - Easy w/ strides


In [ ]:
needs_input = [w for w in enriched if w["classification"]["needs_input"]]
for w in needs_input:
    print(f"{w['date']} | {w.get('garmin_name', '')} | {w.get('name', '')}")

In [ ]:
import sys
sys.path.insert(0, f"{BASE_DIR}/tools")
from parse_workout_data import load_workouts
from calculate_training_load import (
    get_current_metrics, get_recent_trend,
    check_recovery_alert, weekly_load_summary,
    check_mileage_rule, format_metrics_report
)

workouts = load_workouts(
    f"{BASE_DIR}/data/processed/workouts_normalized.json",
    days=180
)

metrics = get_current_metrics(workouts)
trend   = get_recent_trend(workouts, days=14)
alert   = check_recovery_alert(workouts)
mileage = check_mileage_rule(workouts)

print(format_metrics_report(metrics, trend, alert, mileage))

  ℹ️  1 record(s) skipped (no training_load — not Garmin-enriched). Treated as 0 load.
=== Training Load Report — 2026-04-27 ===
  ATL (fatigue):   90.1
  CTL (fitness):   96.1
  TSB (form):      6.1  →  Fresh — good form
  Fitness level:   High fitness

  14-day trend:
    ATL:  123.9 → 90.1  (falling)
    CTL:  97.7 → 96.1  (falling)
    TSB:  -26.2 → 6.1  (rising)
    Total load:     1293
    Avg daily load: 92.3
    Peak load:      320.0
    Rest days:      3 / 14

  Mileage rule: ℹ️  Week just started (day 1/7) — no sessions logged yet. Last week total load: 607. 10% rule target for this week: ≤ 668.


In [ ]:
import sqlite3, sys
sys.path.insert(0, f"{BASE_DIR}/tools")
from query_garmin_db import (
    get_recent_snapshots, get_race_predictions,
    format_recovery_context, format_race_predictions
)

conn = sqlite3.connect(f"{BASE_DIR}/data/raw/garmin/garmin.db")

snapshots = get_recent_snapshots(conn, days=3)
print(format_recovery_context(snapshots))

pred = get_race_predictions(conn)
print(format_race_predictions(pred))

conn.close()

=== Recovery Context ===

--- 2026-04-27 ---

--- 2026-04-26 ---
  Recovery score (composite): 65/100
  Training readiness: 54.0 — Low ⚠️
    Feedback: RECOVERY_IN_PROGRESS
    Recovery time remaining: 1794h
    HRV factor: GOOD
    Sleep history: MODERATE
  HRV: None (weekly avg: 92.0) — Balanced ✅, outside baseline ⚠️
  Sleep: 8.48h total (deep: 1.33h, REM: 1.73h) — Good duration
    Feedback: POSITIVE_LONG_AND_CONTINUOUS
  Body battery at wake: None — No data (peak: None, low: None)
  Stress: avg None, max 96 — Unknown
  Resting HR: 44 bpm

--- 2026-04-25 ---
  Recovery score (composite): 55/100
  Training readiness: 67.0 — Moderate
    Feedback: LISTEN_TO_YOUR_BODY
    Recovery time remaining: 1h
    HRV factor: MODERATE
    Sleep history: GOOD
  HRV: None (weekly avg: 90.0) — Unbalanced ⚠️, outside baseline ⚠️
  Sleep: 6.03h total (deep: 1.27h, REM: 0.73h) — Short sleep ⚠️
    Feedback: POSITIVE_DEEP
  Body battery at wake: None — No data (peak: None, low: None)
  Stress: avg None

In [1]:
%pip install -q chromadb google-generativeai

import sys
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
BASE_DIR       = "/content/drive/MyDrive/running_coach"
GEMINI_API_KEY = userdata.get('key')

sys.path.insert(0, f"{BASE_DIR}/tools")

from memory_retrieval import get_client, retrieve_workouts, retrieve_profile, retrieve_notes, retrieve_all, format_memory_context

# ... rest of the cells

import sys
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
BASE_DIR       = "/content/drive/MyDrive/running_coach"
GEMINI_API_KEY = userdata.get('key')

sys.path.insert(0, f"{BASE_DIR}/tools")

from memory_retrieval import get_client, retrieve_workouts, retrieve_profile, retrieve_notes, retrieve_all, format_memory_context

mem_client, mem_ef = get_client(
    f"{BASE_DIR}/memory/chroma",
    GEMINI_API_KEY
)

# ── Test 1: Workout retrieval ──────────────────────────────────────────────────
print("=== TEST 1: Retrieve similar threshold sessions ===")
results = retrieve_workouts(mem_client, mem_ef, "threshold session pace HR", n=3)
for r in results:
    print(f"\n[{r['metadata']['date']}] dist: {r['distance']:.3f}")
    print(r['document'])

# ── Test 2: Profile retrieval ──────────────────────────────────────────────────
print("\n\n=== TEST 2: Retrieve athlete goals and race context ===")
profile = retrieve_profile(mem_client, mem_ef, "race goal target pace strategy", n=2)
print(profile)

# ── Test 3: Notes retrieval ────────────────────────────────────────────────────
print("\n\n=== TEST 3: Retrieve notes about fatigue or effort ===")
notes = retrieve_notes(mem_client, mem_ef, "tired heavy legs effort", n=2)
if notes:
    for n in notes:
        print(f"[{n['metadata'].get('date','')}] {n['document']}")
else:
    print("No session notes stored yet.")

# ── Test 4: Full context as an agent sees it ───────────────────────────────────
print("\n\n=== TEST 4: Full memory context (as injected into agent prompt) ===")
all_results = retrieve_all(
    mem_client, mem_ef,
    query="how did my recent hard sessions go?",
    n_workouts=3,
    n_profile=2,
    n_notes=2,
)
print(format_memory_context(all_results))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


=== TEST 1: Retrieve similar threshold sessions ===

[2026-04-08] dist: 0.183
Date: 2026-04-08 | Type: vo2max | Distance: 14.48km | Duration: 66.4min
Overall avg pace: 4:35/km | Vo2max target: 3:40/km. Actual: 4:35/km (55s slower than target). Slightly under — check conditions, fatigue, or HR data. ⚠️
HR: 146.8 avg / 174 max | Elevation: 107.0m
Training load: 319.21466064453125 | Aerobic effect: 3.9000000953674316 | Anaerobic effect: 3.5 | Suffer score: 60
Interval splits: Hard efforts: 9 reps, avg 3:31/km, fastest 3:26/km | Recovery/easy laps: 8, avg 6:38/km
Recovery context: HRV status: UNBALANCED, sleep: 7.3h, readiness: 72.0, resting HR: 40.0 bpm
Garmin enriched: yes

[2026-01-14] dist: 0.193
Date: 2026-01-14 | Type: unclassified | Distance: 11.05km | Duration: 48.8min
Overall avg pace: 4:25/km | Cannot evaluate — target pace or actual pace is missing.
HR: 164.3 avg / 185 max | Elevation: 69.0m
Training load: 310.0431823730469 | Aerobic effect: 5.0 | Anaerobic effect: 3.09999990463